## Comparing MMCIF2Dict vs. Bio.PDB.MMCIFParser for parsing speed

**Background**  
One of the core data ingestion steps is to parse structural data from .cif files to obtain alpha-carbon coordinates and other information. There are 3 options each with pros and cons as listed below.

1. MMCIF2Dict 
- direct, raw access to data 
- light weight
- lack of structural topology
2. Bio.PDB.MMCIFParser
- widely used, easy implementation
- slow and memory-intensive

**Goal**  
In this notebook, we will compare the parsing speed of MMCIF2Dict and Bio.PDB.MMCIFParser.

**Hypothesis**  
Because MMCIFParser builds a full structural object tree (Structure $\rightarrow$ Model $\rightarrow$ Chain $\rightarrow$ Residue $\rightarrow$ Atom) while MMCIF2Dict simply parses tokens into a flat Python dictionary, MMCIF2Dict is expected to be significantly faster and consume far less memory. 

In [1]:
import gc
import statistics
import tracemalloc
import time

from pathlib import Path
from Bio.PDB import MMCIF2Dict, MMCIFParser, PDBList


In [2]:
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

input_dir = data_dir / "input"
input_dir.mkdir(parents=True, exist_ok=True)

In [3]:
TEST_CIF_Paht = input_dir/ "8r3y.cif"

In [4]:
if TEST_CIF_Paht.is_file():
    print("8r3y.cif already exists")
    pass
else:
    pdbl = PDBList()

    raw_string_path = pdbl.retrieve_pdb_file(
        "8r3y", pdir=input_dir, file_format="mmCif"
    )

8r3y.cif already exists


In [5]:
def profile_parser_robust(parse_func, file_path, runs=10):
    """Profiles a parsing function over multiple iterations after a warm-up run."""
    times_ms = []
    peak_mbs = []

    # Warm-up run (populates OS disk cache & handles initial module allocations)
    parse_func(file_path)

    for _ in range(runs):
        gc.collect()  # Ensure a clean heap before each run

        tracemalloc.start()
        start_time = time.perf_counter()

        _ = parse_func(file_path)

        end_time = time.perf_counter()
        _, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        times_ms.append((end_time - start_time) * 1000)
        peak_mbs.append(peak_mem / (1024 * 1024))

    return {
        "min_time_ms": min(times_ms),
        "mean_time_ms": statistics.mean(times_ms),
        "median_time_ms": statistics.median(times_ms),
        "std_dev_ms": statistics.stdev(times_ms) if runs > 1 else 0.0,
        "mean_peak_mb": statistics.mean(peak_mbs),
        "median_peak_mb": statistics.median(peak_mbs),
    }


def run_benchmark(runs=10):
    path = Path(TEST_CIF_Paht)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {TEST_CIF_Paht}."
        )

    print(f"--- Benchmarking Parsers on {path.name} ({runs} runs) ---")

    # 1. Benchmark MMCIFParser (Full Structural Tree)
    def parse_with_tree(p):
        parser = MMCIFParser(QUIET=True)
        return parser.get_structure("8R3Y", p)

    stats_tree = profile_parser_robust(parse_with_tree, str(path), runs=runs)
    print("\n[MMCIFParser (Full Object Tree)]")
    print(f"  ├── Min Time:     {stats_tree['min_time_ms']:.2f} ms")
    print(f"  ├── Mean Time:    {stats_tree['mean_time_ms']:.2f} ms")
    print(f"  ├── Median Time:  {stats_tree['median_time_ms']:.2f} ms")
    print(f"  ├── Std Dev:      ±{stats_tree['std_dev_ms']:.2f} ms")
    print(f"  ├── Mean RAM:     {stats_tree['mean_peak_mb']:.2f} MB")
    print(f"  └── Median RAM:   {stats_tree['median_peak_mb']:.2f} MB")

    # 2. Benchmark MMCIF2Dict (Flat Dictionary)
    def parse_with_dict(p):
        return MMCIF2Dict.MMCIF2Dict(p)

    stats_dict = profile_parser_robust(parse_with_dict, str(path), runs=runs)
    print("\n[MMCIF2Dict (Flat Dictionary)]")
    print(f"  ├── Min Time:     {stats_dict['min_time_ms']:.2f} ms")
    print(f"  ├── Mean Time:    {stats_dict['mean_time_ms']:.2f} ms")
    print(f"  ├── Median Time:  {stats_dict['median_time_ms']:.2f} ms")
    print(f"  ├── Std Dev:      ±{stats_dict['std_dev_ms']:.2f} ms")
    print(f"  ├── Mean RAM:     {stats_dict['mean_peak_mb']:.2f} MB")
    print(f"  └── Median RAM:   {stats_dict['median_peak_mb']:.2f} MB")

    # Comparison Summary using Median values
    speedup_median = stats_tree["median_time_ms"] / stats_dict["median_time_ms"]
    speedup_mean = stats_tree["mean_time_ms"] / stats_dict["mean_time_ms"]
    mem_ratio = stats_tree["median_peak_mb"] / stats_dict["median_peak_mb"]

    print("\n" + "=" * 48)
    print("SUMMARY:")
    print(f" • Speedup (by Median): MMCIF2Dict is ~{speedup_median:.1f}x faster")
    print(f" • Speedup (by Mean):   MMCIF2Dict is ~{speedup_mean:.1f}x faster")
    print(f" • RAM Ratio:           MMCIF2Dict uses ~{mem_ratio:.1f}x less RAM")
    print("=" * 48)


In [6]:
run_benchmark(runs=10)

--- Benchmarking Parsers on 8r3y.cif (10 runs) ---

[MMCIFParser (Full Object Tree)]
  ├── Min Time:     3584.55 ms
  ├── Mean Time:    4063.51 ms
  ├── Median Time:  3889.77 ms
  ├── Std Dev:      ±465.57 ms
  ├── Mean RAM:     37.84 MB
  └── Median RAM:   37.84 MB

[MMCIF2Dict (Flat Dictionary)]
  ├── Min Time:     2241.86 ms
  ├── Mean Time:    2325.84 ms
  ├── Median Time:  2322.83 ms
  ├── Std Dev:      ±62.89 ms
  ├── Mean RAM:     17.18 MB
  └── Median RAM:   17.18 MB

SUMMARY:
 • Speedup (by Median): MMCIF2Dict is ~1.7x faster
 • Speedup (by Mean):   MMCIF2Dict is ~1.7x faster
 • RAM Ratio:           MMCIF2Dict uses ~2.2x less RAM
